# 03 — 训练 & 检查模型学到了什么

加载已经训好的权重（运行 `python -m backend.train` 得到的 `models/lenet5.npz`），
然后做几件事：

1. 在测试集上算精度 + 混淆矩阵
2. 可视化 C1 学到的 6 个 5×5 滤波器（人能看懂的话，应该是边缘 / 斑点检测器）
3. 把一张测试图喂进去，画出每层激活
4. 找出模型分错的样本，看看哪些数字最难

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from backend.model import LeNet5
from backend.data import load_mnist_sklearn

model = LeNet5(use_sparse_c3=True)
model.load('../models/lenet5.npz')
print('weights loaded')

_, _, X_te, y_te = load_mnist_sklearn()
print(f'test set: {X_te.shape}')

## 1. 整体精度 + 混淆矩阵

In [ ]:
batch = 256
preds = np.zeros_like(y_te)
for i in range(0, len(X_te), batch):
    preds[i:i+batch] = model.predict(X_te[i:i+batch])
acc = (preds == y_te).mean()
print(f'Test accuracy: {acc*100:.2f}%')

cm = np.zeros((10, 10), dtype=int)
for t, p in zip(y_te, preds):
    cm[t, p] += 1

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(cm, cmap='Blues')
for i in range(10):
    for j in range(10):
        c = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=c, fontsize=9)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_title(f'Confusion matrix (acc = {acc*100:.2f}%)')
plt.tight_layout(); plt.show()

## 2. C1 层学到了什么？
C1 是 6 个 5×5 滤波器，每个都在原图上 "扫描"。直接画 weight。
训练充分的 conv net 这一层通常学的是各方向的边缘检测器或斑点。

In [ ]:
W = model.c1.W   # shape: (6, 1, 5, 5)
fig, axes = plt.subplots(1, 6, figsize=(12, 2.2))
for i in range(6):
    axes[i].imshow(W[i, 0], cmap='RdBu', vmin=-W.max(), vmax=W.max())
    axes[i].set_title(f'C1 filter {i}', fontsize=10)
    axes[i].axis('off')
plt.suptitle('C1 learned filters (red = +, blue = -)', y=1.05)
plt.tight_layout(); plt.show()

## 3. 单张图：所有层激活全景

这正是前端可视化做的事，notebook 里能看到全套数字。

In [ ]:
# 选一张图（你可以改 idx 看不同样本）
idx = 7
x = X_te[idx:idx+1]
true = int(y_te[idx])
logits, acts = model.forward(x, collect_activations=True)
pred = int(np.argmax(logits))
print(f'true={true}, pred={pred}, confidence={acts["probs"][0, pred]*100:.2f}%')

fig = plt.figure(figsize=(16, 8))

# 输入
ax = plt.subplot2grid((4, 9), (0, 0), rowspan=2)
ax.imshow(x[0, 0], cmap='gray', vmin=-1, vmax=1)
ax.set_title(f'Input\n(true={true})'); ax.axis('off')

# C1: 6 个 28x28
for i in range(6):
    ax = plt.subplot2grid((4, 9), (i // 3, 1 + i % 3))
    ax.imshow(acts['C1'][0, i], cmap='inferno')
    ax.set_title(f'C1 #{i}', fontsize=8); ax.axis('off')

# S2: 6 个 14x14
for i in range(6):
    ax = plt.subplot2grid((4, 9), (i // 3, 4 + i % 3))
    ax.imshow(acts['S2'][0, i], cmap='inferno')
    ax.set_title(f'S2 #{i}', fontsize=8); ax.axis('off')

# C3: 16 个 10x10
for i in range(16):
    ax = plt.subplot2grid((4, 9), (2 + i // 8, 1 + i % 8))
    ax.imshow(acts['C3'][0, i], cmap='inferno')
    ax.set_title(f'C3 #{i}', fontsize=7); ax.axis('off')

# Output probabilities
ax = plt.subplot2grid((4, 9), (0, 7), rowspan=2, colspan=2)
colors = ['#34d399' if i == pred else '#38bdf8' for i in range(10)]
ax.barh(range(10), acts['probs'][0], color=colors)
ax.set_yticks(range(10))
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_title('Output probabilities')

plt.tight_layout(); plt.show()

## 4. 看看分错的样本
看哪些数字最容易让模型犯傻。一般是 4↔9、3↔5、7↔1 这些视觉上相近的对。

In [ ]:
wrong = np.where(preds != y_te)[0]
print(f'{len(wrong)} mistakes out of {len(y_te)}')

rng = np.random.default_rng(0)
show = rng.choice(wrong, min(20, len(wrong)), replace=False)

fig, axes = plt.subplots(4, 5, figsize=(10, 8))
for ax, k in zip(axes.flat, show):
    ax.imshow(X_te[k, 0], cmap='gray', vmin=-1, vmax=1)
    ax.set_title(f'true={y_te[k]} pred={preds[k]}', fontsize=10, color='#dc2626')
    ax.axis('off')
plt.suptitle('Misclassified examples', y=1.005)
plt.tight_layout(); plt.show()

## 5. 哪两个类最容易混？

In [ ]:
# 把对角线 (正确的) 清零，找最大的几个 off-diagonal
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
pairs = []
for i in range(10):
    for j in range(10):
        if i != j and cm_off[i, j] > 0:
            pairs.append((cm_off[i, j], i, j))
pairs.sort(reverse=True)
print('Top 10 confusion pairs (true -> predicted):')
for n, t, p in pairs[:10]:
    print(f'  {t} -> {p}: {n} times')